# Portfolio Note

This notebook preserves the original research workflow. Raw Taaghche data, intermediate spreadsheets, the pre-labeled gold annotation file, and row-level prediction files are intentionally not included in the public portfolio version. Aggregate results are available under `results/`.


In [1]:
import torch
import pandas as pd
import numpy as np

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 3070


In [2]:
DATA_PATH = "taaghche_absa_aspect_detection_final_clean.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig"
)

print(df.shape)
df.head()

(70410, 11)


,book_id,comment_id,text_original,text,rating,likes,dislikes,author,detected_aspects,aspect_count,has_detected_aspect
0,72,233742,عرض ادب. علی (ع) معجزه ای است که خداوند او را ...,عرض ادب. علی (ع) معجزه‌ای است که خداوند او را ...,5.0,262,10,مرتضی مطهری,[],0,False
1,72,362407,اینقدر زوایای وجودی و شخصیتی امیرالمومنین ع نا...,اینقدر زوایای وجودی و شخصیتی امیرالمومنین ع نا...,5.0,89,6,مرتضی مطهری,[],0,False
2,72,411806,بیانی روان و مطالبی مفید از استاد شهید.\n**\nب...,بیانی روان و مطالبی مفید از استاد شهید. ** به ...,5.0,53,2,مرتضی مطهری,"[""author""]",1,True
3,72,678736,کتاب جالبی بود؛ مخصوصا نیمه دوم کتاب! بحث مربو...,کتاب جالبی بود؛ مخصوصا نیمه دوم کتاب! بحث مربو...,5.0,42,2,مرتضی مطهری,[],0,False
4,72,700060,خیلی عالیه...\nبه نظرم حتما یک بار بخونیدش...\...,خیلی عالیه … به نظرم حتما یک‌بار بخونیدش … خیل...,5.0,33,4,مرتضی مطهری,[],0,False


In [3]:
df.columns.tolist()

['book_id',
 'comment_id',
 'text_original',
 'text',
 'rating',
 'likes',
 'dislikes',
 'author',
 'detected_aspects',
 'aspect_count',
 'has_detected_aspect']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70410 entries, 0 to 70409
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   book_id              70410 non-null  int64  
 1   comment_id           70410 non-null  int64  
 2   text_original        70410 non-null  str    
 3   text                 70410 non-null  str    
 4   rating               69638 non-null  float64
 5   likes                70410 non-null  int64  
 6   dislikes             70410 non-null  int64  
 7   author               69935 non-null  str    
 8   detected_aspects     70410 non-null  str    
 9   aspect_count         70410 non-null  int64  
 10  has_detected_aspect  70410 non-null  bool   
dtypes: bool(1), float64(1), int64(5), str(4)
memory usage: 45.1 MB


In [5]:
df.isna().sum()

book_id                  0
comment_id               0
text_original            0
text                     0
rating                 772
likes                    0
dislikes                 0
author                 475
detected_aspects         0
aspect_count             0
has_detected_aspect      0
dtype: int64

In [6]:
df["detected_aspects"].value_counts().head(20)

detected_aspects
[]                                      39474
["story"]                               10912
["content"]                              4317
["author"]                               2726
["story", "author"]                      2286
["story", "content"]                     1701
["translation"]                           979
["story", "content", "author"]            616
["translation", "story"]                  609
["content", "author"]                     594
["story", "writing_style", "author"]      563
["writing_style"]                         511
["story", "writing_style"]                489
["writing_style", "author"]               431
["audio_narrator"]                        391
["price"]                                 382
["translation", "content"]                255
["story", "audio_narrator"]               251
["translation", "story", "author"]        196
["translation", "author"]                 155
Name: count, dtype: int64

In [7]:
df["aspect_count"].value_counts()

aspect_count
0    39474
1    20552
2     7506
3     2272
4      499
5       87
6       19
7        1
Name: count, dtype: int64

In [8]:
import ast

df["aspect_list"] = df["detected_aspects"].apply(ast.literal_eval)

print(df["aspect_list"].head())

0          []
1          []
2    [author]
3          []
4          []
Name: aspect_list, dtype: object


In [9]:
# فقط کامنت‌هایی که حداقل یک aspect دارند

df_aspect = df[df["aspect_count"] > 0].copy()

print(df_aspect.shape)
df_aspect.head()

(30936, 12)


,book_id,comment_id,text_original,text,rating,likes,dislikes,author,detected_aspects,aspect_count,has_detected_aspect,aspect_list
2,72,411806,بیانی روان و مطالبی مفید از استاد شهید.\n**\nب...,بیانی روان و مطالبی مفید از استاد شهید. ** به ...,5.0,53,2,مرتضی مطهری,"[""author""]",1,True,[author]
6,72,264769,این کتاب رو تاکنون دو سه بار خوانده ام . استاد...,این کتاب رو تاکنون دو سه بار خوانده‌ام. استاد ...,5.0,25,2,مرتضی مطهری,"[""author""]",1,True,[author]
7,72,918984,راستش تصورم از این کتاب یه چیز دیگه بود،فکر می...,راستش تصورم از این کتاب یه چیز دیگه بود، فکر م...,5.0,23,1,مرتضی مطهری,"[""author""]",1,True,[author]
8,72,341178,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,4.0,24,2,مرتضی مطهری,"[""author""]",1,True,[author]
9,72,416343,رحم الله امرء عرف قدره...\nخدا رحمت کند بنده ا...,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای ...,5.0,21,3,مرتضی مطهری,"[""price""]",1,True,[price]


In [10]:
from tqdm import tqdm

rows = []

for _, row in df_aspect.iterrows():
    for aspect in row["aspect_list"]:
        rows.append({
            "comment_id": row["comment_id"],
            "book_id": row["book_id"],
            "text": row["text"],
            "text_original": row["text_original"],
            "aspect": aspect,
            "rating": row["rating"],
            "likes": row["likes"],
            "dislikes": row["dislikes"],
            "author": row["author"]
        })

aspect_context_df = pd.DataFrame(rows)

print(aspect_context_df.shape)

aspect_context_df.head()

(44932, 9)


,comment_id,book_id,text,text_original,aspect,rating,likes,dislikes,author
0,411806,72,بیانی روان و مطالبی مفید از استاد شهید. ** به ...,بیانی روان و مطالبی مفید از استاد شهید.\n**\nب...,author,5.0,53,2,مرتضی مطهری
1,264769,72,این کتاب رو تاکنون دو سه بار خوانده‌ام. استاد ...,این کتاب رو تاکنون دو سه بار خوانده ام . استاد...,author,5.0,25,2,مرتضی مطهری
2,918984,72,راستش تصورم از این کتاب یه چیز دیگه بود، فکر م...,راستش تصورم از این کتاب یه چیز دیگه بود،فکر می...,author,5.0,23,1,مرتضی مطهری
3,341178,72,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,author,4.0,24,2,مرتضی مطهری
4,416343,72,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای ...,رحم الله امرء عرف قدره...\nخدا رحمت کند بنده ا...,price,5.0,21,3,مرتضی مطهری


In [11]:
aspect_context_df["aspect"].value_counts()

aspect
story             18938
content            8722
author             8478
translation        3267
writing_style      2818
audio_narrator      882
price               612
publisher           552
censorship          377
print_editing       205
cover_design         81
Name: count, dtype: int64

In [12]:
aspect_context_df["rating"].isna().sum()

np.int64(273)

In [13]:
aspect_context_df["rating"].value_counts(dropna=False)

rating
5.0    26322
4.0     7883
3.0     5339
1.0     2764
2.0     2351
NaN      273
Name: count, dtype: int64

In [14]:
# پیدا کردن کامنت‌های تک اسپکتی

single_ids = df[df["aspect_count"] == 1]["comment_id"]

single_context_df = aspect_context_df[
    aspect_context_df["comment_id"].isin(single_ids)
].copy()

multi_context_df = aspect_context_df[
    ~aspect_context_df["comment_id"].isin(single_ids)
].copy()


print("Single aspect:", single_context_df.shape)
print("Multi aspect:", multi_context_df.shape)

Single aspect: (20552, 9)
Multi aspect: (24380, 9)


In [15]:
def rating_to_sentiment(r):
    if pd.isna(r):
        return None
    elif r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:
        return "positive"


single_context_df["sentiment_label"] = (
    single_context_df["rating"]
    .apply(rating_to_sentiment)
)

single_context_df["sentiment_label"].value_counts(dropna=False)

sentiment_label
positive    16129
negative     2222
neutral      2082
NaN           119
Name: count, dtype: int64

In [17]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

single_context_model_df = single_context_df.dropna(
    subset=["sentiment_label"]
).copy()

single_context_model_df["sentiment_id"] = (
    single_context_model_df["sentiment_label"]
    .map(label_map)
)

print(single_context_model_df.shape)

single_context_model_df.head()

(20433, 11)


,comment_id,book_id,text,text_original,aspect,rating,likes,dislikes,author,sentiment_label,sentiment_id
0,411806,72,بیانی روان و مطالبی مفید از استاد شهید. ** به ...,بیانی روان و مطالبی مفید از استاد شهید.\n**\nب...,author,5.0,53,2,مرتضی مطهری,positive,2
1,264769,72,این کتاب رو تاکنون دو سه بار خوانده‌ام. استاد ...,این کتاب رو تاکنون دو سه بار خوانده ام . استاد...,author,5.0,25,2,مرتضی مطهری,positive,2
2,918984,72,راستش تصورم از این کتاب یه چیز دیگه بود، فکر م...,راستش تصورم از این کتاب یه چیز دیگه بود،فکر می...,author,5.0,23,1,مرتضی مطهری,positive,2
3,341178,72,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,به نظرم یکی از بهترین کتاب‌های شهید مطهری هست....,author,4.0,24,2,مرتضی مطهری,positive,2
4,416343,72,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای ...,رحم الله امرء عرف قدره...\nخدا رحمت کند بنده ا...,price,5.0,21,3,مرتضی مطهری,positive,2


In [18]:
from sklearn.model_selection import train_test_split

X = single_context_model_df["text"]
y = single_context_model_df["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

(16346,)
(4087,)

Train distribution:
sentiment_label
positive    0.789367
negative    0.108712
neutral     0.101921
Name: proportion, dtype: float64

Test distribution:
sentiment_label
positive    0.789332
negative    0.108882
neutral     0.101786
Name: proportion, dtype: float64


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1,2),
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

tfidf_lr.fit(
    X_train,
    y_train
)

print("Training finished")

Training finished


In [20]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

y_pred_lr = tfidf_lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nMacro F1:", f1_score(
    y_test,
    y_pred_lr,
    average="macro"
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred_lr
))

Accuracy: 0.7797895767066307

Macro F1: 0.6016555549422469

Classification Report:
              precision    recall  f1-score   support

    negative       0.52      0.60      0.56       445
     neutral       0.30      0.45      0.36       416
    positive       0.93      0.85      0.88      3226

    accuracy                           0.78      4087
   macro avg       0.58      0.63      0.60      4087
weighted avg       0.82      0.78      0.80      4087


Confusion Matrix:
[[ 269  103   73]
 [  85  188  143]
 [ 165  331 2730]]


In [21]:
from sklearn.svm import LinearSVC

tfidf_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1,2),
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LinearSVC(
            class_weight="balanced",
            random_state=42
        )
    )
])


tfidf_svm.fit(
    X_train,
    y_train
)

print("SVM training finished")

SVM training finished


In [22]:
y_pred_svm = tfidf_svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nMacro F1:", f1_score(
    y_test,
    y_pred_svm,
    average="macro"
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred_svm
))

Accuracy: 0.818693418155126

Macro F1: 0.5855090839220153

Classification Report:
              precision    recall  f1-score   support

    negative       0.60      0.54      0.57       445
     neutral       0.33      0.24      0.28       416
    positive       0.89      0.93      0.91      3226

    accuracy                           0.82      4087
   macro avg       0.61      0.57      0.59      4087
weighted avg       0.80      0.82      0.81      4087


Confusion Matrix:
[[ 241   66  138]
 [  79  100  237]
 [  81  140 3005]]


In [23]:
import pandas as pd

baseline_results = pd.DataFrame([
    {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": 0.7797895767,
        "macro_f1": 0.6016555549
    },
    {
        "model": "TF-IDF + Linear SVM",
        "accuracy": 0.8186934182,
        "macro_f1": 0.5855090839
    }
])

baseline_results

,model,accuracy,macro_f1
0,TF-IDF + Logistic Regression,0.779790,0.601656
1,TF-IDF + Linear SVM,0.818693,0.585509


In [24]:
baseline_results.to_csv(
    "baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved ✅")

Saved ✅


In [25]:
multi_context_df["aspect"].value_counts()

aspect
story             8026
author            5752
content           4405
writing_style     2307
translation       2288
audio_narrator     491
publisher          430
censorship         248
price              230
print_editing      147
cover_design        56
Name: count, dtype: int64

In [26]:
multi_context_df["rating"].value_counts(dropna=False)

rating
5.0    13320
4.0     4756
3.0     3257
2.0     1461
1.0     1432
NaN      154
Name: count, dtype: int64

In [27]:
import pandas as pd

context_df = pd.read_excel(
    "taaghche_aspect_contexts.xlsx"
)

print(context_df.shape)
print(context_df.columns)

context_df.head()

(46221, 4)
Index(['comment_id', 'book_id', 'aspect', 'context'], dtype='str')


,comment_id,book_id,aspect,context
0,416343,72,price,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای ...
1,1276036,72,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم ...
2,1276036,72,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین ...
3,2129147,72,content,به نام خدا؛ بنظرم کتاب‌های آقای مطهری بلااستثن...
4,2129147,72,story,شخصیت کامل حضرت علی (ع) نمیپردازه. بلکه فقط خص...


In [28]:
import pandas as pd
import numpy as np
import json
import re

pd.set_option('display.max_colwidth', 300)


with open(
    "taaghche_frozen_aspect_rules.json",
    "r",
    encoding="utf-8"
) as f:
    frozen_rules = json.load(f)


ASPECT_PATTERNS = {
    aspect: re.compile(pattern)
    for aspect, pattern in frozen_rules["aspect_patterns"].items()
}


AUTHOR_PATTERN = re.compile(
    frozen_rules["author_generic_pattern"]
)

In [29]:
def extract_aspect_anchors(text):

    anchors = []


    for aspect, pattern in ASPECT_PATTERNS.items():

        for match in pattern.finditer(text):

            anchors.append(
                {
                    "aspect": aspect,
                    "anchor": match.group(),
                    "start": match.start(),
                    "end": match.end()
                }
            )


    # author rule
    for match in AUTHOR_PATTERN.finditer(text):

        anchors.append(
            {
                "aspect": "author",
                "anchor": match.group(),
                "start": match.start(),
                "end": match.end()
            }
        )


    return anchors

In [30]:
def get_sorted_aspect_anchors(text):

    return sorted(
        extract_aspect_anchors(text),
        key=lambda x: x["start"]
    )



def merge_same_aspect_anchors(anchors):

    if not anchors:
        return []


    merged = []

    current = anchors[0].copy()


    for anchor in anchors[1:]:

        if anchor["aspect"] == current["aspect"]:

            current["end"] = anchor["end"]

        else:

            merged.append(current)
            current = anchor.copy()


    merged.append(current)

    return merged



def get_merged_aspect_anchors(text):

    anchors = get_sorted_aspect_anchors(text)

    return merge_same_aspect_anchors(anchors)

In [31]:
BOUNDARY_WORDS = [
    "ولی",
    "اما",
    "با این حال"
]


def clean_context_boundary(context):

    context = context.strip()


    for word in BOUNDARY_WORDS:

        if context.endswith(word):

            context = context[:-len(word)].strip()


    return context

In [32]:
def extract_aspect_contexts_v3(text):

    anchors = get_merged_aspect_anchors(text)


    if len(anchors) == 0:

        return []


    contexts = []


    for i, anchor in enumerate(anchors):


        if i == 0:

            start = 0

        else:

            start = anchor["start"]



        if i < len(anchors)-1:

            end = anchors[i+1]["start"]

        else:

            end = len(text)



        context = text[start:end].strip()


        contexts.append(
            {
                "aspect": anchor["aspect"],
                "context": clean_context_boundary(context)
            }
        )


    return contexts

In [33]:
tests = [
    "ترجمه خوب بود داستان عالی بود",
    "نویسنده عالی بود ولی داستان ضعیف بود"
]


for t in tests:

    print("\nTEXT:")
    print(t)

    print("\nCONTEXTS:")

    for c in extract_aspect_contexts_v3(t):

        print(
            f"- {c['aspect']}: {c['context']}"
        )


TEXT:
ترجمه خوب بود داستان عالی بود

CONTEXTS:
- translation: ترجمه خوب بود
- story: داستان عالی بود

TEXT:
نویسنده عالی بود ولی داستان ضعیف بود

CONTEXTS:
- author: نویسنده عالی بود
- story: داستان ضعیف بود


In [34]:
from tqdm import tqdm

tqdm.pandas()


context_rows = []

for idx, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    text = row["text"]

    contexts = extract_aspect_contexts_v3(text)

    for item in contexts:

        context_rows.append(
            {
                "comment_id": row["comment_id"],
                "book_id": row["book_id"],
                "aspect": item["aspect"],
                "context": item["context"]
            }
        )


context_df = pd.DataFrame(context_rows)


print(context_df.shape)

context_df.head()

100%|██████████| 70410/70410 [01:50<00:00, 637.72it/s] 


(46472, 4)


,comment_id,book_id,aspect,context
0,416343,72,price,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای که قدر و قیمت خودش را می‌شناسد. کاربران گرامی: راجع به این کلمات بنویسیم.
1,1276036,72,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...
2,1276036,72,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...
3,2129147,72,content,به نام خدا؛ بنظرم کتاب‌های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند. کتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی (ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...
4,2129147,72,story,شخصیت کامل حضرت علی (ع) نمیپردازه. بلکه فقط خصوصیات جذب و دفع ایشون رو مورد بررسی قرار میده، پس برای کسانی که به دنبال مطالعه زندگی کامل و شخصیت مولاعلی (ع) هستن کتاب‌های دیگه‌ای مثل «فروغ ولایت»، «علی از زبان علی» و … رو پیشنهاد می‌کنم. یاعلی مدد


In [45]:
def is_valid_context(context):
    
    if context is None:
        return False
    
    context = context.strip()
    
    if len(context) < 3:
        return False
    
    words = context.split()
    
    if len(words) == 0:
        return False
    
    return True


context_df = context_df[
    context_df["context"].apply(is_valid_context)
].reset_index(drop=True)


print(context_df.shape)

(46221, 4)


In [46]:
context_df["aspect"].value_counts()

aspect
story             21571
content            9439
author             6122
translation        3527
writing_style      2733
audio_narrator      914
price               629
publisher           598
censorship          393
print_editing       214
cover_design         81
Name: count, dtype: int64

In [35]:
context_df["aspect"].value_counts()

aspect
story             21597
content            9440
author             6122
translation        3527
writing_style      2957
audio_narrator      914
price               629
publisher           598
censorship          393
print_editing       214
cover_design         81
Name: count, dtype: int64

In [36]:
context_df.isna().sum()

comment_id    0
book_id       0
aspect        0
context       0
dtype: int64

In [37]:
context_df.to_csv(
    "taaghche_aspect_contexts_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [38]:
old_context_df = pd.read_excel(
    "taaghche_aspect_contexts.xlsx"
)

print(old_context_df.shape)
print(context_df.shape)

(46221, 4)
(46472, 4)


In [39]:
old_context_df["aspect"].value_counts()

aspect
story             21571
content            9439
author             6122
translation        3527
writing_style      2733
audio_narrator      914
price               629
publisher           598
censorship          393
print_editing       214
cover_design         81
Name: count, dtype: int64

In [40]:
context_df["aspect"].value_counts()

aspect
story             21597
content            9440
author             6122
translation        3527
writing_style      2957
audio_narrator      914
price               629
publisher           598
censorship          393
print_editing       214
cover_design         81
Name: count, dtype: int64

In [41]:
old_ws = old_context_df[
    old_context_df["aspect"]=="writing_style"
]

new_ws = context_df[
    context_df["aspect"]=="writing_style"
]


print(old_ws.shape)
print(new_ws.shape)

(2733, 4)
(2957, 4)


In [42]:
merged = new_ws.merge(
    old_ws,
    on=["comment_id","book_id","aspect","context"],
    how="left",
    indicator=True
)

merged[merged["_merge"]=="left_only"].head(20)

,comment_id,book_id,aspect,context,_merge
85,2460218,882,writing_style,,left_only
124,1307841,1176,writing_style,,left_only
126,400171,1176,writing_style,,left_only
148,627109,3754,writing_style,,left_only
149,645433,3754,writing_style,از,left_only
153,647846,3754,writing_style,,left_only
178,877358,3754,writing_style,,left_only
187,2340862,3754,writing_style,,left_only
201,1000832,3754,writing_style,,left_only
202,1000832,3754,writing_style,,left_only


In [47]:
context_df.to_csv(
    "taaghche_aspect_contexts_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [48]:
# اضافه کردن rating از دیتاست اصلی

context_df = context_df.merge(
    df[
        [
            "comment_id",
            "rating",
            "text_original"
        ]
    ],
    on="comment_id",
    how="left"
)

context_df.head()

,comment_id,book_id,aspect,context,rating,text_original
0,416343,72,price,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای که قدر و قیمت خودش را می‌شناسد. کاربران گرامی: راجع به این کلمات بنویسیم.,5.0,رحم الله امرء عرف قدره...\nخدا رحمت کند بنده ای که قدر و قیمت خودش را می شناسد.\nکاربران گرامی:راجع به این کلمات بنویسیم.
1,1276036,72,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...,5.0,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...
2,1276036,72,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...,5.0,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...
3,2129147,72,content,به نام خدا؛ بنظرم کتاب‌های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند. کتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی (ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...,5.0,به نام خدا؛ بنظرم کتاب های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند.\nکتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی(ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...
4,2129147,72,story,شخصیت کامل حضرت علی (ع) نمیپردازه. بلکه فقط خصوصیات جذب و دفع ایشون رو مورد بررسی قرار میده، پس برای کسانی که به دنبال مطالعه زندگی کامل و شخصیت مولاعلی (ع) هستن کتاب‌های دیگه‌ای مثل «فروغ ولایت»، «علی از زبان علی» و … رو پیشنهاد می‌کنم. یاعلی مدد,5.0,به نام خدا؛ بنظرم کتاب های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند.\nکتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی(ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...


In [49]:
context_df.shape

(46221, 6)

In [50]:
context_df["rating"].isna().sum()

np.int64(268)

In [51]:
aspect_per_comment = (
    context_df
    .groupby("comment_id")["aspect"]
    .count()
)


single_ids = aspect_per_comment[
    aspect_per_comment == 1
].index


multi_ids = aspect_per_comment[
    aspect_per_comment > 1
].index


single_context_df = context_df[
    context_df["comment_id"].isin(single_ids)
].copy()


multi_context_df = context_df[
    context_df["comment_id"].isin(multi_ids)
].copy()


print("Single:", single_context_df.shape)
print("Multi:", multi_context_df.shape)

Single: (20473, 6)
Multi: (25748, 6)


In [52]:
single_context_df["rating"].isna().sum()

np.int64(118)

In [53]:
single_context_df["rating"].value_counts(dropna=False)

rating
5.0    12841
4.0     3152
3.0     2140
1.0     1324
2.0      898
NaN      118
Name: count, dtype: int64

In [54]:
def rating_to_sentiment(r):

    if pd.isna(r):
        return None
    
    if r <= 2:
        return "negative"
    
    elif r == 3:
        return "neutral"
    
    else:
        return "positive"


single_context_df["sentiment_label"] = (
    single_context_df["rating"]
    .apply(rating_to_sentiment)
)

In [55]:
single_context_df["sentiment_label"].value_counts(dropna=False)

sentiment_label
positive    15993
negative     2222
neutral      2140
NaN           118
Name: count, dtype: int64

In [56]:
single_sentiment_df = single_context_df[
    single_context_df["sentiment_label"].notna()
].copy()


print(single_sentiment_df.shape)

single_sentiment_df["sentiment_label"].value_counts()

(20355, 7)


sentiment_label
positive    15993
negative     2222
neutral      2140
Name: count, dtype: int64

In [57]:
single_sentiment_df[
    [
        "aspect",
        "context",
        "sentiment_label"
    ]
].head(10)

,aspect,context,sentiment_label
0,price,رحم الله امرء عرف قدره … خدا رحمت کند بنده‌ای که قدر و قیمت خودش را می‌شناسد. کاربران گرامی: راجع به این کلمات بنویسیم.,positive
5,content,سلام کتابی همراه با نکاتی بسیار عالی و آموزنده به همراه تحلیلی از وقایع زمان امام علی (ع),positive
6,content,متن قابل‌فهم و مطالب جذابی داره، نوجوونهای کتابخون، حتما از خوندنش فایده می‌برن,positive
8,content,مطالب کتاب درباره اهمیت جاذبه و دافعه، نقش محبت در دین اسلام و در ادامه در قسمت دافعه علی (ع) بیشتر معطوف به ماجرای خوارج است. در کل مطالب مهمی گفته‌شده که در تاریخ تکرار شده,positive
9,content,عالی، دلنشین، جذاب و پرمعنا,positive
10,content,بنظرم خیلی کتاب خوبی بود. ارزش خوندن داره. فقط انتظار داشتم که خیلی بیشتر درباره امام علی گفته باشه، اما بنظرم خلاصه‌تر بود. اما مطالبی که درباره خوارج بود بسیار آموزنده بود و جای تفکر و تامل داره,positive
11,price,عالی بود ممنون از طاقچه انصافا چقدر دین اسلام زیباست فقط من حدود یک ماه پیش این کتابو گرفتم رایگان بود یعنی الان باید پول واریز کنم؟؟,positive
12,writing_style,دیدگاهی کامل از مفهوم جذب و دفع به انسان می‌دهد به خوبی مفاهیمی همچون عشق را توصیف کرده و به زوایای مختلف آن پرداخته البته تعریف بنده حقیر چیزی به ارزش قلم استاد مطهری اضاف نمی‌کند,positive
13,content,علی که بود و که هست و علی چه کرد و چه شد؟ علیست جزو سوالات بی‌جواب هنوز با اشاره و ثابت کردن شهید مطهری از نگاه فلسفی و عرفانی که هر آدمی جاذبه و دافعه داره و نگاه بسیار بسیار زیباشون به عشق و دوست داشتن و همچنین اگر دوست دارین بیشتر در مورد جنگ‌های امیرالمومنین بدونید و همیشه یه علامت سوال تو ذ...,positive
14,writing_style,به نظر من شهید استاد مطهری اندیشمندی هستند که به دین اسلام از دید یک انسان به دنبال حقیقت نگاه می‌کنند و همین دید باعث شده قلمشون نه از سر احساس و نگاه معنوی باشه که کاملا قلم حقیقت‌جو باشه من این کتاب رو پیشنهاد می‌کنم با این کتاب متوجه شدم که اسلام چقدر به انسان‌های بلند نظر و صاحب فکر نیاز داره,positive


In [58]:
X = single_sentiment_df["context"]
y = single_sentiment_df["sentiment_label"]

In [59]:
stratify=y

In [60]:
from sklearn.model_selection import train_test_split


X = single_sentiment_df["context"]
y = single_sentiment_df["sentiment_label"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)


print("\nTrain distribution:")
print(y_train.value_counts(normalize=True))


print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

(16284,)
(4071,)

Train distribution:
sentiment_label
positive    0.785679
negative    0.109187
neutral     0.105134
Name: proportion, dtype: float64

Test distribution:
sentiment_label
positive    0.785802
negative    0.109064
neutral     0.105134
Name: proportion, dtype: float64


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)


tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,2)
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


tfidf_lr.fit(
    X_train,
    y_train
)


y_pred = tfidf_lr.predict(X_test)


print("Accuracy:",
      accuracy_score(y_test, y_pred))


print("\nMacro F1:",
      f1_score(
          y_test,
          y_pred,
          average="macro"
      ))


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

Accuracy: 0.7654139032178826

Macro F1: 0.5789469445723009

Classification Report:
              precision    recall  f1-score   support

    negative       0.48      0.58      0.53       444
     neutral       0.28      0.41      0.33       428
    positive       0.92      0.84      0.88      3199

    accuracy                           0.77      4071
   macro avg       0.56      0.61      0.58      4071
weighted avg       0.80      0.77      0.78      4071


Confusion Matrix:
[[ 257   97   90]
 [ 104  174  150]
 [ 174  340 2685]]


In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)


tfidf_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,2)
        )
    ),
    (
        "clf",
        LinearSVC(
            class_weight="balanced",
            random_state=42
        )
    )
])


tfidf_svm.fit(
    X_train,
    y_train
)


y_pred_svm = tfidf_svm.predict(X_test)


print("Accuracy:",
      accuracy_score(y_test, y_pred_svm))


print("\nMacro F1:",
      f1_score(
          y_test,
          y_pred_svm,
          average="macro"
      ))


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_svm
    )
)


print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_svm
    )
)

Accuracy: 0.807909604519774

Macro F1: 0.5607316229404312

Classification Report:
              precision    recall  f1-score   support

    negative       0.57      0.52      0.54       444
     neutral       0.29      0.20      0.24       428
    positive       0.88      0.93      0.90      3199

    accuracy                           0.81      4071
   macro avg       0.58      0.55      0.56      4071
weighted avg       0.78      0.81      0.79      4071


Confusion Matrix:
[[ 230   61  153]
 [  90   84  254]
 [  84  140 2975]]


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)


tfidf_lr_tuned = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=100000,
            ngram_range=(1,3),
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


tfidf_lr_tuned.fit(
    X_train,
    y_train
)


y_pred_tuned = tfidf_lr_tuned.predict(X_test)


print("Accuracy:",
      accuracy_score(y_test, y_pred_tuned))


print("\nMacro F1:",
      f1_score(
          y_test,
          y_pred_tuned,
          average="macro"
      ))


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_tuned
    )
)


print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_tuned
    )
)

Accuracy: 0.7710636207320068

Macro F1: 0.5817239047544449

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      0.56      0.53       444
     neutral       0.28      0.41      0.33       428
    positive       0.92      0.85      0.88      3199

    accuracy                           0.77      4071
   macro avg       0.57      0.61      0.58      4071
weighted avg       0.81      0.77      0.79      4071


Confusion Matrix:
[[ 250  107   87]
 [ 102  174  152]
 [ 148  336 2715]]


In [64]:
baseline_results = pd.DataFrame([
    {
        "model": "TFIDF + Logistic Regression",
        "accuracy": 0.7654139,
        "macro_f1": 0.5789469
    },
    {
        "model": "TFIDF + Linear SVM",
        "accuracy": 0.8079096,
        "macro_f1": 0.5607316
    },
    {
        "model": "TFIDF + Logistic Regression Tuned",
        "accuracy": 0.7710636,
        "macro_f1": 0.5817239
    }
])


baseline_results

,model,accuracy,macro_f1
0,TFIDF + Logistic Regression,0.765414,0.578947
1,TFIDF + Linear SVM,0.807910,0.560732
2,TFIDF + Logistic Regression Tuned,0.771064,0.581724


In [65]:
baseline_results.to_csv(
    "baseline_results_context_based.csv",
    index=False,
    encoding="utf-8-sig"
)

In [66]:
multi_aspect_distribution = (
    multi_context_df
    .groupby("comment_id")["aspect"]
    .count()
    .value_counts()
    .sort_index()
)


multi_aspect_distribution

aspect
2     5307
3     2316
4      871
5      341
6      173
7      100
8       45
9       27
10      20
11      10
12       8
13       6
14       4
15       4
16       1
17       1
23       1
Name: count, dtype: int64

In [67]:
multi_annotation_df = multi_context_df[
    [
        "comment_id",
        "text_original",
        "aspect",
        "context"
    ]
].copy()


multi_annotation_df.head()

,comment_id,text_original,aspect,context
1,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...
2,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...
3,2129147,به نام خدا؛ بنظرم کتاب های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند.\nکتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی(ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...,content,به نام خدا؛ بنظرم کتاب‌های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند. کتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی (ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...
4,2129147,به نام خدا؛ بنظرم کتاب های آقای مطهری بلااستثنا و به معنای واقعی مفید و سودمند هستند.\nکتاب بسیار خوب و نافعی بود. این کتاب همانطور که از اسمش پیداست، به عوامل جاذبه و دافعه حضرت علی(ع) میپردازه و بخش اول در مورد جاذبه و دوستان علی و بخش دوم در مورد دافعه و دشمنان علی یعنی خوارج هست که بنظرم بخش...,story,شخصیت کامل حضرت علی (ع) نمیپردازه. بلکه فقط خصوصیات جذب و دفع ایشون رو مورد بررسی قرار میده، پس برای کسانی که به دنبال مطالعه زندگی کامل و شخصیت مولاعلی (ع) هستن کتاب‌های دیگه‌ای مثل «فروغ ولایت»، «علی از زبان علی» و … رو پیشنهاد می‌کنم. یاعلی مدد
15,2566499,دوست داشتنی و جذاب !\nقلم استاد مطهری هم که خیلی خوب و روون بود در کتاب توضیح میده که امیر المومنین علیهماسلام علاوه بر جاذبه دافعه نیز داشتند که البته ویژگی هر فرد و انسان سالمی ، وجود دافعه است,writing_style,دوست‌داشتنی و جذاب! قلم استاد مطهری هم که خیلی خوب و روون بود در کتاب


In [68]:
multi_annotation_df.shape

(25748, 4)

In [70]:
multi_comments_df = (
    multi_annotation_df
    .groupby("comment_id")
    .agg(
        {
            "text_original": "first",
            "aspect": list,
            "context": list
        }
    )
    .reset_index()
)


multi_comments_df["aspect_count"] = (
    multi_comments_df["aspect"].apply(len)
)


multi_comments_df.head()

,comment_id,text_original,aspect,context,aspect_count
0,1326,تمامی کتابهای شهید مطهری مفیده،\nتوصیه من به عنوان برادر کوچک و بخصوص جوانان اینه که از دستشون ندهند از اینکه رایگان در این برنامه قرار داده شده است متشکرم.,"[content, price]","[تمامی کتابهای شهید مطهری مفیده، توصیه من به عنوان برادر کوچک و بخصوص جوانان اینه که از دستشون ندهند از اینکه, رایگان در این برنامه قرار داده‌شده است متشکرم.]",2
1,1454,بهترین رمانی که توی عمرم خوندم با نویسنده همگام شده بود و تمام فضا به طرز عجیبی در ذهنم تداعی شد این از اون رمان هایی هستش که تا قبل از مرگ باید خوند خدا رو شکر که لیاقت خوندنشو داشتم,"[story, author]","[بهترین رمانی که توی عمرم خوندم با, نویسنده همگام شده بود و تمام فضا به طرز عجیبی در ذهنم تداعی شد این از اون رمان‌هایی هستش که تا قبل از مرگ باید خوند خدا رو شکر که لیاقت خوندنشو داشتم]",2
2,1691,متن گیرا و قلم پرکششی داره. اما مجموعا به نظر میاد هدایت افکار پوچگرایی و ضدباورانه ی زیادی داشته و حسب توصیف اهل فن به خاطر تمول مالی و رفاه کامل، به اشباع و نهایتا پوچی رسیده.\nاما واقعا داستان خوشنویسیه. خصوصا که همذات پنداری زیادی در خیلی جاهای داستان برای مخاطب پیش میاد که همراهی با متن رو ...,"[writing_style, story, content]","[متن گیرا و قلم پرکششی داره. اما مجموعا به نظر میاد هدایت افکار پوچگرایی و ضدباورانه‌ی زیادی داشته و حسب توصیف اهل فن به خاطر تمول مالی و رفاه کامل، به اشباع و نهایتا پوچی رسیده. اما واقعا, داستان خوشنویسیه. خصوصا که همذات پنداری زیادی در خیلی جاهای داستان برای مخاطب پیش میاد که همراهی با متن رو...",3
3,2079,تصویر سازی فوق العاده یعنی هرچی که نوشته شده تو کتاب رو خیلی راحت تصور میکنین. یه کشش و جذب کنندگی ی خیلی خوب که وادار میکنه تا آخر کتاب رو چند روزه بخونین . تامل برانگیز و کمی هم ناراحت کننده ولی به شدت جذاب .حتما بخونین,"[story, content]","[تصویر سازی فوق‌العاده یعنی هرچی که نوشته‌شده تو کتاب رو خیلی راحت تصور میکنین. یه کشش و جذب کنندگی‌ی خیلی خوب که وادار میکنه تا آخر کتاب رو چند روزه بخونین., تامل برانگیز و کمی هم ناراحت‌کننده ولی به شدت جذاب. حتما بخونین]",2
4,2167,من این رمان رو تا حالا ده بار خوندم. و شگفتی قلم هدایت اینه که بازم میخوام بخونمش و بازم ازش برداشت های جدید می کنم. این کتاب در تاریخ ادبیات ما جاودانی میمونه,"[story, writing_style]","[من این رمان رو تا حالا ده بار خوندم. و شگفتی, قلم هدایت اینه که بازم میخوام بخونمش و بازم ازش برداشت‌های جدید می‌کنم. این کتاب در تاریخ ادبیات ما جاودانی میمونه]",2


In [71]:
multi_comments_df.shape

(9235, 5)

In [72]:
multi_comments_df["aspect_count"].value_counts().sort_index()

aspect_count
2     5307
3     2316
4      871
5      341
6      173
7      100
8       45
9       27
10      20
11      10
12       8
13       6
14       4
15       4
16       1
17       1
23       1
Name: count, dtype: int64

In [73]:
gold_samples = pd.concat([
    multi_comments_df[
        multi_comments_df["aspect_count"] == 2
    ].sample(
        n=500,
        random_state=42
    ),

    multi_comments_df[
        multi_comments_df["aspect_count"] == 3
    ].sample(
        n=300,
        random_state=42
    ),

    multi_comments_df[
        multi_comments_df["aspect_count"] == 4
    ].sample(
        n=100,
        random_state=42
    ),

    multi_comments_df[
        multi_comments_df["aspect_count"] >= 5
    ].sample(
        n=100,
        random_state=42
    )
])


gold_samples = gold_samples.reset_index(drop=True)


print(gold_samples.shape)

gold_samples["aspect_count"].value_counts()

(1000, 5)


aspect_count
2     500
3     300
4     100
5      47
6      26
7      12
8       5
9       3
11      2
15      2
13      1
14      1
10      1
Name: count, dtype: int64

In [74]:
gold_annotation_df = gold_samples.copy()


gold_annotation_df["sentiment_labels"] = None


gold_annotation_df.head()

,comment_id,text_original,aspect,context,aspect_count,sentiment_labels
0,712753,تالستوی نویسنده بزرگی است اگر آثار ایشان به درستی ترجمه شوند ارزش بارها خواندن را داراست....لذت بردم...ممنون طاقچه,"[author, translation]","[تالستوی نویسنده بزرگی است اگر آثار ایشان به درستی, ترجمه شوند ارزش بارها خواندن را داراست …. لذت بردم …ممنون طاقچه]",2,None
1,2868990,ایده اولیه کتاب خیلی جالب بود، اما در ادامه داستان اونقدری کشش نداشت.\nدر کل میخواست به ادم یاداوری کنه که هر روز از زندگی چقدر با اهمیته...,"[content, story]","[ایده اولیه کتاب خیلی جالب بود، اما در ادامه, داستان اونقدری کشش نداشت. در کل می‌خواست به ادم یاداوری کنه که هر روز از زندگی چقدر با اهمیته …]",2,None
2,1107789,برعکس نظر خیلی های دیگه به نظر من این کتاب خیلی خیلی می تونست بهتر باشه . به نظر من ایده این کتاب بسیار جذاب است اما به طور حوصله سر بر و آرومی روایت شده . به طوری که از هر جای کتاب شروع به خوندن کنی متوجه کلیت داستان می شی . 😶,"[content, story]","[برعکس نظر خیلی‌های دیگه به نظر من این کتاب خیلی خیلی می‌تونست بهتر باشه. به نظر من ایده این کتاب بسیار جذاب است اما به طور حوصله سر بر و آرومی روایت شده. به طوری که از هر جای کتاب شروع به خوندن کنی متوجه کلیت, داستان می‌شی.]",2,None
3,868328,من یه ترجمه دیگه از این کتاب رو خوندم و در این رابطه نمیتونم نظر بدم ولی در رابطه با محتوای کتاب باید بگم که لحظه به لحظه با ویکتور فرانکل تو اردوگاه آشویتس زندگی کردم،،گاهی دردهاشو تا عمق قلبم حس کردم و گاهی هرچی تلاش کردم چیزی حس نکردم و به ضعف خودم پی بردم.\nتناقض حس جذاب و عمیقیه پس بهترین ز...,"[translation, content]","[من یه ترجمه دیگه از این کتاب رو خوندم و در این رابطه نمیتونم نظر بدم ولی در رابطه با, محتوای کتاب باید بگم که لحظه به لحظه با ویکتور فرانکل تو اردوگاه آشویتس زندگی کردم،، گاهی دردهاشو تا عمق قلبم حس کردم و گاهی هرچی تلاش کردم چیزی حس نکردم و به ضعف خودم پی بردم. تناقض حس جذاب و عمیقیه پس بهترین...",2,None
4,1687427,سلام به شما من مدت ها بود که کتاب نخونده بودم یا کتاب ها رو نصفه می خوندم چون اون جذابیت رو برام نداشتن که ادامه شون بدم ولی این کتاب یه چیز دیگه بود یه تلفیقی از رمان و روانشناسی که بنظرم من در هر صورت عالی و بی نقصه یه کتاب هدف دار و انگیزشی شاید اول که شروع به خواندن کنید اینجوری به نظر نرسه ...,"[story, content]","[سلام به شما من مدت‌ها بود که کتاب نخونده بودم یا کتاب‌ها رو نصفه می‌خوندم چون اون جذابیت رو برام نداشتن که ادامه شون بدم ولی این کتاب یه چیز دیگه بود یه تلفیقی از رمان و روانشناسی که بنظرم من در هر صورت عالی و بی نقصه یه کتاب هدف‌دار و, انگیزشی شاید اول که شروع به خواندن کنید اینجوری به نظر نرس...",2,None


In [75]:
gold_annotation_df.columns

Index(['comment_id', 'text_original', 'aspect', 'context', 'aspect_count',
       'sentiment_labels'],
      dtype='str')

In [76]:
gold_annotation_df.to_csv(
    "gold_aspect_annotation_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

In [77]:
gold_context_df = multi_annotation_df[
    multi_annotation_df["comment_id"].isin(
        gold_samples["comment_id"]
    )
].copy()


print(gold_context_df.shape)

gold_context_df.head()

(2931, 4)


,comment_id,text_original,aspect,context
1,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...
2,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...
24,2126497,بسیار عالی مفید و پر از مطالب جدید.قلم خیلی روان و جذاب,content,بسیار عالی مفید و پر از مطالب جدید.
25,2126497,بسیار عالی مفید و پر از مطالب جدید.قلم خیلی روان و جذاب,writing_style,قلم خیلی روان و جذاب
95,2488429,من چون داشتم از پادکست امیر خادم گوش میدادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب تر هست.\nاما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست.\nموضوعی که متاسفانه بابتش غبطه می خورم این هست که در کشور تاجیکستان به هر خانواده دو...,censorship,من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور تاجیکستان به هر خانواده دو ...


In [78]:
gold_context_df["sentiment_label"] = ""

In [79]:
gold_context_df.head()

,comment_id,text_original,aspect,context,sentiment_label
1,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...,
2,1276036,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه ای که حضرت برای بعضی افراد داره\nکتاب دو بخش داره\nتو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه\nای...,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...,
24,2126497,بسیار عالی مفید و پر از مطالب جدید.قلم خیلی روان و جذاب,content,بسیار عالی مفید و پر از مطالب جدید.,
25,2126497,بسیار عالی مفید و پر از مطالب جدید.قلم خیلی روان و جذاب,writing_style,قلم خیلی روان و جذاب,
95,2488429,من چون داشتم از پادکست امیر خادم گوش میدادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب تر هست.\nاما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست.\nموضوعی که متاسفانه بابتش غبطه می خورم این هست که در کشور تاجیکستان به هر خانواده دو...,censorship,من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور تاجیکستان به هر خانواده دو ...,


In [80]:
gold_context_df.to_csv(
    "gold_aspect_annotation.csv",
    index=False,
    encoding="utf-8-sig"
)

In [81]:
gold_context_df.info()

<class 'pandas.DataFrame'>
Index: 2931 entries, 1 to 46200
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   comment_id       2931 non-null   int64
 1   text_original    2931 non-null   str  
 2   aspect           2931 non-null   str  
 3   context          2931 non-null   str  
 4   sentiment_label  2931 non-null   str  
dtypes: int64(1), str(4)
memory usage: 3.4 MB


In [82]:
gold_context_df["sentiment_label"].value_counts(dropna=False)

sentiment_label
    2931
Name: count, dtype: int64

In [83]:
gold_context_df.sample(
    10,
    random_state=42
)

,comment_id,text_original,aspect,context,sentiment_label
21094,1834278,قصر آبی، اثری از نویسنده مجموعه کتاب های آن شرلی که احتمالا حتی اگر کسی آن هارا نخوانده باشد از معروف و مشهور بودن آن اطلاع دارد.حقیقتا ابتدای کتاب داشتم به این فکر می کردم که چقدر به عنوان یک رمان کلاسیک در مقایسه ذهنی ام با آثار جین آستن، خواهران برونته و سایر کلاسیک های این مدلی با چه شاهکاری...,story,رمان کلاسیک در مقایسه ذهنی‌ام با آثار جین آستن، خواهران برونته و سایر کلاسیک‌های این مدلی با چه شاهکاری روبرو هستم و چقدر مانند آن شرلی دارم با ولنسی (شخص اول) ارتباط می‌گیرم. جملات شاهکار کتاب را واقعا نمی‌شود نادیده گرفت و بیشترشان را هایلایت کردم و معتقدم چندین قدم از این جهت و سایر جملات کلی...,
37703,2738324,اولین باره که در این سبک و ژانر کتابی می‌خوندم، بدم نیومد اما داستانی نیست که به دیگران پیشنهاد بدم چون هیجان خاصی برام نداشت و همه چی روشن بود :) از نظر ادبی و متن خوبه اما داستان ویژه و غیرمنتظره‌ای نداشت.,story,اولین باره که در این سبک و ژانر کتابی می‌خوندم، بدم نیومد اما داستانی نیست که به دیگران پیشنهاد بدم چون هیجان خاصی برام نداشت و همه چی روشن بود:),
44603,2229887,اولین کتابیِ که با ژانر جنایی خوندم!\nواقعا انتظار نداشتم که انقدر غیر قابل پیش‌بینی باشه.\nو احساس میکنم با خوندن این کتاب واقعا به قلم مک‌فادان علاقه مند شدم و دلم میخواد تمام کتاب های این نویسنده رو بخونم.\nاونقدر جذاب بود که هیچ جوره نمیشد ولش کنی و تو دوروز تمومش کردم. همه جوره پیشنهاد میشه...,story,اولین کتابی که با ژانر جنایی خوندم! واقعا انتظار نداشتم که انقدر غیر قابل پیش‌بینی باشه. و احساس می‌کنم با خوندن این کتاب واقعا به,
33867,983367,این کتاب چند شاخصه داشت اول که رمان مستند از زندگی حقیقی یک شخص هست که معاصر هست و قابل تحقیق بوده و به گفته ناشر صحت ادعای راوی اثبات شده دوم اینکه مطالبش مطابقت زیادی با قران و احادیث داره و من لا اقل ندیدم جایی تضاد داشته باشه سوم مورد تایید علمای موضوع هم بوده و قبل چاپ تایید محتوایی شده چها...,content,مطالبش مطابقت زیادی با قران و احادیث داره و من لا اقل ندیدم جایی تضاد داشته باشه سوم مورد تایید علمای موضوع هم بوده و قبل چاپ تایید محتوایی شده چهارم خیلی ساده و روان نوشته‌شده و مخاطب رو دنبال خودش میکشونه و تصویر سازی‌های بی نظیری داره پنجم مختصر و مفید هست و از بیان مطالب اضافی پرهیز شده البت...,
23659,2269531,داستانش به نظرم خیلی سطحی و الکی طولانی بود ولی در کل خوب بود ولی برای تایمی که میزارید نمی ارزه به نظرم گوش دادنش کل داستان رو میشه توی پنج صفحه خلاصه کرد .,story,داستانش به نظرم خیلی,
38370,1561104,از اون دسته کتابایی بود که بعد از تمام کردنش، یه تجربه ی جدید و ارزشمند بهت هدیه میده.\nشما نه تنها این کتاب رو میخونید بلکه زندگی میکنیدش. به شخصه خودم خیلی به این موضوع فکر کردم. اگر فقط یک روز به پایان عمرم مانده بود، چه کارهایی میکردم؟ البته به نتیجه ای نرسیدم... به نظرم بیشتر شبیه متیو میشد...,story,از اون دسته کتابایی بود که بعد از تمام کردنش، یه تجربه‌ی جدید و ارزشمند بهت هدیه میده. شما نه‌تنها این کتاب رو میخونید بلکه زندگی میکنیدش. به‌شخصه خودم خیلی به این موضوع فکر کردم. اگر فقط یک روز به پایان عمرم مانده بود، چه کارهایی می‌کردم؟ البته به نتیجه‌ای نرسیدم … به نظرم بیشتر شبیه متیو می‌شد...,
35638,2166384,سلام کتاب خیلی خوبی هست داستان آموزنده ای داره فقط بنظرم زیادی داستان کش اومد و از وسط کتاب آدم\nدیگه کم کم خسته میشه البته شاید من اینطور بودم... ولی ارزش خوندن داره و در انتهای کتاب جمع بندی و نتیجه گیری داستان عالی هست و میشه گفت اصل کل کتاب همون چند صفحه اس ...,story,سلام کتاب خیلی خوبی هست داستان,
6866,2426511,- این کتاب مجموعه‌ای از احساس‌های مختلف رو به شما تزریق می‌کنه. بعضی صفحات شما رو غرق در خنده می‌کنه و برخی دیگه بغض عجیبی در گلوتون‌ میکاره‌. روند کلی کتاب تا حدی میشه گفت خطی پیش رفته اما کشش‌ خوبی داره و مشتاق‌تون‌ می‌کنه تمومش‌ کنید..\n- در ضمن فیلمش هم خوش ساخت و بامزه‌ست اما مسلما جزئیاتش ...,translation,مترجم که خیلیی زیبا متن رو برگردوندن، من با ترجمه‌ی دیگه‌ای هم مطالعه کردم ولی به کیفیت کار خانم فرحناکیان نمی‌رسه. در کل خوندن این کتاب جالب و پند آموز بود و پیشنهاد می‌کنم حتما براش وقت بزارید.,
12867,2423451,مطلب اموزنده ندارد. داستان ساده ای است.,story,داستان ساده‌ای است.,
7250,1814729,از خواندنش لذت بردم،از غافلگیری های نویسنده، از توصیف واضح و بیش از حد افراد و موقعیت ها و حرکات و صحنه، از حضور یک زن

In [84]:
gold_context_df["annotator_note"] = ""

In [85]:
gold_context_df.to_csv(
    "gold_aspect_annotation_ready_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [86]:
gold_excel_df = gold_context_df.copy()


# اضافه کردن ستون یادداشت برای annotation
gold_excel_df["annotator_note"] = ""


# انتخاب ستون‌های مناسب برای کار دستی
gold_excel_df = gold_excel_df[
    [
        "comment_id",
        "aspect",
        "context",
        "sentiment_label",
        "annotator_note"
    ]
]


gold_excel_df.head()

,comment_id,aspect,context,sentiment_label,annotator_note
1,1276036,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...,,
2,1276036,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...,,
24,2126497,content,بسیار عالی مفید و پر از مطالب جدید.,,
25,2126497,writing_style,قلم خیلی روان و جذاب,,
95,2488429,censorship,من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور تاجیکستان به هر خانواده دو ...,,


In [87]:
gold_excel_df.to_excel(
    "gold_aspect_annotation_for_labeling.xlsx",
    index=False
)

In [88]:
import pandas as pd

gold_df = pd.read_excel(
    "gold_aspect_annotation_prelabelled.xlsx"
)

gold_df.head()

,comment_id,aspect,context,sentiment_label,annotator_note
0,1276036,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...,neutral,pre-labeled
1,1276036,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...,positive,pre-labeled
2,2126497,content,بسیار عالی مفید و پر از مطالب جدید.,positive,pre-labeled
3,2126497,writing_style,قلم خیلی روان و جذاب,positive,pre-labeled
4,2488429,censorship,من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور تاجیکستان به هر خانواده دو ...,positive,pre-labeled


In [89]:
gold_df.shape

(2931, 5)

In [90]:
# ساخت ورودی ParsBERT

gold_df["model_input"] = (
    "[ASPECT] "
    + gold_df["aspect"].astype(str)
    + " [SEP] "
    + gold_df["context"].astype(str)
)

gold_df[
    [
        "aspect",
        "context",
        "model_input",
        "sentiment_label"
    ]
].head()

,aspect,context,model_input,sentiment_label
0,story,امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش دوم هم از دافعه اینکه...,[ASPECT] story [SEP] امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش ...,neutral
1,content,توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات خوبی بده برای نیمه دوم...,[ASPECT] content [SEP] توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات...,positive
2,content,بسیار عالی مفید و پر از مطالب جدید.,[ASPECT] content [SEP] بسیار عالی مفید و پر از مطالب جدید.,positive
3,writing_style,قلم خیلی روان و جذاب,[ASPECT] writing_style [SEP] قلم خیلی روان و جذاب,positive
4,censorship,من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور تاجیکستان به هر خانواده دو ...,[ASPECT] censorship [SEP] من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور ت...,positive


In [91]:
# تبدیل sentiment به عدد

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}


gold_df["labels"] = (
    gold_df["sentiment_label"]
    .map(label_map)
)


gold_df[
    [
        "model_input",
        "sentiment_label",
        "labels"
    ]
].head()

,model_input,sentiment_label,labels
0,[ASPECT] story [SEP] امام علی یک شخصیت دو نیرویی، هم جاذبه داره هم دافعه. جاذبه و دافعه بسیار زیادی هم داره وقتی بهش جذب میشن یکی میشه مثل میثم و از اون طرف کسایی که دشمن سرسختش میشن به خاطر دافعه‌ای که حضرت برای بعضی افراد داره کتاب دو بخش داره تو بخش اول از نیروی جاذبه علی صحبت میکنه و در بخش ...,neutral,1
1,[ASPECT] content [SEP] توضیح میده و مثال میزنه بعد خوندن متوجه میشین که چرا بعد رحلت پیامبر مردم به دسته‌های مختلف و در مقابل هم قرار گرفتن چرا بعضی‌ها در مقابل امام که پیامبر اون رو به عنوان جانشین انتخاب کرد قبول نداشتن نمی‌شد از کل اتفاقات و جاذبه‌ها و … بگه (کتاب الغارات میتونه بهتون اطلاعات...,positive,2
2,[ASPECT] content [SEP] بسیار عالی مفید و پر از مطالب جدید.,positive,2
3,[ASPECT] writing_style [SEP] قلم خیلی روان و جذاب,positive,2
4,[ASPECT] censorship [SEP] من چون داشتم از پادکست امیر خادم گوش می‌دادم برای همین خواستم ابیات رو هم داشته باشم اینطوری برام جذاب‌تر هست. اما متاسفانه این کتاب حذفیات زیادی داره، ابیاتش جابجا شدند و بسیاری از ابیاتش هم با اصلش متفاوت هست. موضوعی که متاسفانه بابتش غبطه می‌خورم این هست که در کشور ت...,positive,2


In [92]:
gold_df["labels"].value_counts()

labels
1    1475
2    1179
0     277
Name: count, dtype: int64

In [93]:
from sklearn.model_selection import train_test_split


train_df, temp_df = train_test_split(
    gold_df,
    test_size=0.2,
    random_state=42,
    stratify=gold_df["labels"]
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["labels"]
)


print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (2344, 7)
Validation: (293, 7)
Test: (294, 7)


In [94]:
train_df["labels"].value_counts(normalize=True)
val_df["labels"].value_counts(normalize=True)
test_df["labels"].value_counts(normalize=True)

labels
1    0.503401
2    0.401361
0    0.095238
Name: proportion, dtype: float64

In [95]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3070


In [96]:
!pip install -q transformers datasets accelerate sentencepiece

In [97]:
import transformers
import datasets

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

C:\Users\ASUS\miniconda3\envs\taaghche_absa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.15.1
datasets: 5.0.1


In [98]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_name = "HooshvareLab/bert-base-parsbert-uncased"


tokenizer = AutoTokenizer.from_pretrained(model_name)


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)


print("Tokenizer and model loaded successfully")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6167.96it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differ

Tokenizer and model loaded successfully


In [99]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

cuda


In [100]:
from datasets import Dataset


# تبدیل pandas به HuggingFace Dataset
train_dataset = Dataset.from_pandas(
    train_df[["model_input", "labels"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["model_input", "labels"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["model_input", "labels"]]
)


print(train_dataset)

Dataset({
    features: ['model_input', 'labels', '__index_level_0__'],
    num_rows: 2344
})


In [101]:
def tokenize_function(batch):
    return tokenizer(
        batch["model_input"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


print(train_dataset)

Map: 100%|██████████| 294/294 [00:00<00:00, 3809.62 examples/s]

Dataset({
    features: ['model_input', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2344
})


In [102]:
train_dataset = train_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)

val_dataset = val_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)

test_dataset = test_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)


print(train_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2344
})


In [103]:
train_dataset = train_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)

val_dataset = val_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)

test_dataset = test_dataset.remove_columns(
    ["model_input", "__index_level_0__"]
)


print(train_dataset)

ValueError: Column name ['model_input', '__index_level_0__'] not in the dataset. Current columns in the dataset: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [104]:
train_dataset.set_format(
    "torch"
)

val_dataset.set_format(
    "torch"
)

test_dataset.set_format(
    "torch"
)

print(train_dataset[0])

{'labels': tensor(1), 'input_ids': tensor([    2,    24, 81808, 20573,    26, 50522,     4,  4556,  2773,  4363,
         2083,   331,  3517,  3100,  3018,  2036,  2389, 39563,  1176,  4123,
         2083,    15,     4,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [105]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight


class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["labels"]),
    y=train_df["labels"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)


class_weights

tensor([3.5354, 0.6621, 0.8286], device='cuda:0')

In [108]:
from transformers import Trainer


class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights
        )

        loss = loss_fct(
            logits.view(-1, 3),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [109]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = logits.argmax(axis=-1)

    acc = accuracy_score(
        labels,
        predictions
    )

    f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": acc,
        "macro_f1": f1
    }

In [111]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./parsbert_absa_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=50,

    fp16=True,

    report_to="none"
)

In [113]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    processing_class=tokenizer
)

In [114]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.710618,0.402226,0.863481,0.814173
2,0.396830,0.351177,0.901024,0.873436
3,0.233023,0.309041,0.911263,0.884015


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.19s/it]


TrainOutput(global_step=879, training_loss=0.4714352992343143, metrics={'train_runtime': 149.6614, 'train_samples_per_second': 46.986, 'train_steps_per_second': 5.873, 'total_flos': 462553388365824.0, 'train_loss': 0.4714352992343143, 'epoch': 3.0})

In [115]:
test_results = trainer.evaluate(
    test_dataset
)

test_results

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.233023,0.445836,3,0.877551,0.846647


{'eval_loss': 0.4458364248275757,
 'eval_accuracy': 0.8775510204081632,
 'eval_macro_f1': 0.8466468848541826}

In [116]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


pred_output = trainer.predict(test_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(
    pred_output.predictions,
    axis=1
)


print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "negative",
            "neutral",
            "positive"
        ]
    )
)


print("Confusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

    negative       0.70      0.82      0.75        28
     neutral       0.93      0.82      0.87       148
    positive       0.87      0.96      0.91       118

    accuracy                           0.88       294
   macro avg       0.83      0.87      0.85       294
weighted avg       0.88      0.88      0.88       294

Confusion Matrix:
[[ 23   5   0]
 [  9 122  17]
 [  1   4 113]]


In [117]:
model_save_path = "./taaghche_parsbert_absa_final"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Model saved:", model_save_path)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.34s/it]

Model saved: ./taaghche_parsbert_absa_final


In [118]:
model_save_path = "./taaghche_parsbert_absa_final"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.22s/it]


('./taaghche_parsbert_absa_final\\tokenizer_config.json',
 './taaghche_parsbert_absa_final\\tokenizer.json')

In [119]:
import numpy as np
import pandas as pd

# prediction روی test
pred_output = trainer.predict(test_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)


# برگرداندن label متنی
label_names = {
    0: "negative",
    1: "neutral",
    2: "positive"
}


results_df = test_df.copy()

results_df["true_label"] = [
    label_names[x] for x in y_true
]

results_df["predicted_label"] = [
    label_names[x] for x in y_pred
]


results_df.to_csv(
    "parsbert_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


results_df.head()

,comment_id,aspect,context,sentiment_label,annotator_note,model_input,labels,true_label,predicted_label
192,2206121,story,داستان درباره‌ی دختری ۱۵ ساله به نام سوفی است که نامه‌های اسرارآمیزی دریافت می‌کند. این نامه‌ها شامل سوالات,negative,pre-labeled,[ASPECT] story [SEP] داستان درباره‌ی دختری ۱۵ ساله به نام سوفی است که نامه‌های اسرارآمیزی دریافت می‌کند. این نامه‌ها شامل سوالات,0,negative,negative
1430,988414,story,رمان جذابن ولی در سیر خط اصلی داستان خیلی پیشرفت زیادی حاصل نمیشه.,positive,pre-labeled,[ASPECT] story [SEP] رمان جذابن ولی در سیر خط اصلی داستان خیلی پیشرفت زیادی حاصل نمیشه.,2,positive,positive
2787,2568831,story,شخصیت‌پردازی شخصیت اصلیه که داستان از زبان اون روایت میشه ولی به حدی به عنوان یک کتاب ترسناک، ترسناک نیست که اصلا زبان,neutral,pre-labeled,[ASPECT] story [SEP] شخصیت‌پردازی شخصیت اصلیه که داستان از زبان اون روایت میشه ولی به حدی به عنوان یک کتاب ترسناک، ترسناک نیست که اصلا زبان,1,neutral,neutral
1865,1388251,content,پر مفهوم نوشته به نظرم,neutral,pre-labeled,[ASPECT] content [SEP] پر مفهوم نوشته به نظرم,1,neutral,neutral
2899,1906357,author,نویسنده که پزشک متبحری هس برای علاقه مندان به مساله تروما از منظر خوندن تجربیات این پزشک، مناسبه,neutral,pre-labeled,[ASPECT] author [SEP] نویسنده که پزشک متبحری هس برای علاقه مندان به مساله تروما از منظر خوندن تجربیات این پزشک، مناسبه,1,neutral,neutral


In [120]:
from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(
    y_true,
    y_pred,
    target_names=[
        "negative",
        "neutral",
        "positive"
    ],
    output_dict=True
)


report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    "parsbert_classification_report.csv",
    encoding="utf-8-sig"
)


report_df

,precision,recall,f1-score,support
negative,0.696970,0.821429,0.754098,28.000000
neutral,0.931298,0.824324,0.874552,148.000000
positive,0.869231,0.957627,0.911290,118.000000
accuracy,0.877551,0.877551,0.877551,0.877551
macro avg,0.832499,0.867793,0.846647,294.000000
weighted avg,0.884070,0.877551,0.877826,294.000000


In [121]:
from sklearn.metrics import confusion_matrix


cm = confusion_matrix(
    y_true,
    y_pred
)


cm_df = pd.DataFrame(
    cm,
    index=["negative_true","neutral_true","positive_true"],
    columns=["negative_pred","neutral_pred","positive_pred"]
)


cm_df.to_csv(
    "parsbert_confusion_matrix.csv",
    encoding="utf-8-sig"
)


cm_df

,negative_pred,neutral_pred,positive_pred
negative_true,23,5,0
neutral_true,9,122,17
positive_true,1,4,113


In [122]:
history_df = pd.DataFrame(
    trainer.state.log_history
)


history_df.to_csv(
    "parsbert_training_history.csv",
    index=False,
    encoding="utf-8-sig"
)


history_df.tail()

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_macro_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
17,0.253323,0.441371,1.820250e-06,2.730375,800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,0.233023,175.048157,6.825939e-07,2.901024,850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,NaN,NaN,NaN,3.000000,879,0.309041,0.911263,0.884015,0.9079,322.730,20.928,NaN,NaN,NaN,NaN,NaN
20,NaN,NaN,NaN,3.000000,879,NaN,NaN,NaN,NaN,NaN,NaN,149.6614,46.986,5.873,4.625534e+14,0.471435
21,NaN,NaN,NaN,3.000000,879,0.445836,0.877551,0.846647,1.0087,291.454,18.835,NaN,NaN,NaN,NaN,NaN
